# Упражнение 4: логистична регресия

**Машинно самообучение · зимен семестър 2026/2027**

**Repo:** https://github.com/mkorudzhiev/ML_26_27<br>
**Discord:** https://discord.gg/WKztMqX46

Мартин Коруджиев<br/>
mkorudzhiev@uni-sofia.bg<br/>
0885 71 31 60

## Съдържание

1. [Цел на упражнението](#cel)
2. [Файлове в упражнението](#failove)
3. [Подготовка и стартиране](#podgotovka)
4. [Част 1. Класификация](#chast-1)
5. [Част 2. Перцептрон](#chast-2)
6. [Част 3. Логистична регресия](#chast-3)
7. [Част 4. Шансове и logit](#chast-4)
8. [Част 5. Максимално правдоподобие](#chast-5)
9. [Част 6. Вероятност при два признака](#chast-6)
10. [Част 7. Многокласова класификация](#chast-7)
11. [Обобщение](#obobshtenie)
12. [Задача](#zadacha)

<a id="cel"></a>
## Цел на упражнението

В упражнение 3 линейната регресия предсказваше число. Сега отговорът е
**клас**: спам или не, постоянна позиция или не, кой вид ирис. Такава
задача се нарича класификация.

Упражнението отговаря на шест въпроса:

| Въпрос | Къде |
|---|---|
| Как линеен модел може да избира клас? | части 1 и 2 |
| Защо линейната регресия не е подходяща за класове 0 и 1? | част 3 |
| Какво означава теглото в логистичната регресия? | част 4 |
| Как се намират теглата? | част 5 |
| Как изглежда границата между класовете? | части 1, 2 и 6 |
| Какво се прави при повече от два класа? | част 7 |

Основният пример е малка измислена таблица с 14 преподаватели: колко
години стаж има всеки, колко публикации има и дали има постоянна позиция.
Таблицата е малка, за да може всяка сметка да се провери на ръка. Другите
примери са познати: ирисите от упражнение 2, част 1, и SMS съобщенията от
упражнение 2, част 4. Примерите следват лекцията.

<a id="failove"></a>
## Файлове в упражнението

Пътищата са относителни спрямо папката на тази тетрадка.

| Файл | За какво служи |
|---|---|
| `Упражнение_4.ipynb` | обяснения, примери и задачи |
| `ЗА_ВКЪЩИ.ipynb` | задачи за самостоятелна работа |
| `solution.py` | решения на задачите ДА ОПИТАМЕ и на финалната задача |
| `data/sms_spam.csv` | 5574 SMS съобщения, означени като spam или legit (за задачата) |
| `data/default.csv` | 10 000 клиенти: неплатен дълг по кредитна карта, студент, баланс и доход (за домашното) |
| `../environment.yml` | обща среда при работа с `conda` |
| `../requirements.txt` | общ списък с пакети при работа с `pip` |

Ирисите се зареждат с `load_iris` от scikit-learn, както в упражнение 2,
част 1. `sms_spam.csv` е същият файл като в упражнение 2, част 4.
`default.csv` е Default.csv от [сайта на авторите на *An Introduction to
Statistical Learning*](https://www.statlearning.com/resources-python),
основният пример за логистична регресия в глава 4 на книгата.

<a id="podgotovka"></a>
## Подготовка и стартиране

Използваме средата, създадена по време на първото упражнение.
От папката на **четвъртото** упражнение обновяваме пакетите по
избрания начин, ако е необходимо. Първо активираме средата:

| Среда | Команди |
|---|---|
| Anaconda Prompt | 1. `conda activate ml2627`<br/>2. `conda env update -f ../environment.yml` |
| PowerShell и `pip` | 1. `..\.venv\Scripts\Activate.ps1`<br/>2. `python -m pip install -r ../requirements.txt` |

След това, от същата папка, стартираме `jupyter lab`, отваряме
`Упражнение_4.ipynb` и избираме ядрото **Python (МС 26/27)**.
Ако пакет се намира в терминала, но липсва в тетрадката, проверяваме
избраното ядро. Ако CSV файлът не се намира, проверяваме от коя папка
сме стартирали Jupyter.

In [ ]:
import solution

%load_ext autoreload
%autoreload 2

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display

print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("работна папка:", os.getcwd())

<a id="chast-1"></a>
## Част 1. Класификация

<a id="t1"></a>
### 1. Класове вместо числа

При **класификацията** моделът не предсказва число, а избира **клас** от
няколко възможни. Например спам филтърът получава имейл и отговаря "спам"
или "не е спам".

Когато класовете са два, те се означават с 1 и 0. Класът 1 се нарича
**положителен**, а класът 0 **отрицателен**. Това не е оценка: при спам
филтъра положителният клас е спамът. Положителен е просто класът, който се
търси.

Примерът в тази част са 14 преподаватели. За всеки е известно:

- `years`: колко години стаж има;
- `publications`: колко публикации има;
- `tenured`: 1, ако има постоянна позиция (tenure), и 0, ако няма.

In [ ]:
tenure = pd.DataFrame({
    "years":        [1, 3, 3, 3, 4, 5, 5, 6, 7, 8, 9, 9, 11, 12],
    "publications": [4, 2, 5, 18, 10, 6, 8, 12, 3, 9, 7, 11, 6, 5],
    "tenured":      [0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1],
})
print("Преподаватели:", len(tenure))
print(tenure["tenured"].value_counts().rename({0: "без позиция (0)", 1: "с позиция (1)"}))
display(tenure.T)

<a id="t2"></a>
### 2. Граница на решението

Класификаторът разделя всички възможни входове на области, по една за
всеки клас. Линията между областите се нарича **граница на решението**
(decision boundary). Видът ѝ зависи от броя на признаците:

- при един признак границата е точка;
- при два признака е права;
- при три е равнина, а при повече се нарича хиперравнина.

Първо само стажът. Най-простото правило е "постоянна позиция при поне 6
години стаж", т.е. границата е точката 6. Следващата клетка проверява
колко преподаватели познава това правило. На графиката преподавателите с
еднакъв стаж и клас са леко изместени един над друг, за да се виждат
всички.

In [ ]:
def stacked(frame):
    """Класът 0 или 1, леко изместен за повтарящите се точки."""
    offset = frame.groupby(["years", "tenured"]).cumcount() * 0.06
    return frame["tenured"] + np.where(frame["tenured"] == 1, -offset, offset)


rule = (tenure["years"] >= 6).astype(int)
print("Правилни според правилото:", int((rule == tenure["tenured"]).sum()), "от", len(tenure))
display(tenure[rule != tenure["tenured"]])

fig, ax = plt.subplots(figsize=(8, 2.8))
ax.scatter(tenure["years"], stacked(tenure), s=60, zorder=3,
           c=np.where(tenure["tenured"] == 1, "#2a78d6", "#eb6834"))
ax.axvline(6, color="black", linestyle="--", linewidth=1, label="граница: 6 години")
ax.set(xlabel="години стаж", ylabel="tenured", yticks=[0, 1], title="Постоянна позиция според стажа")
ax.legend(loc="center right")
plt.show()

Правилото греши за двама преподаватели: един с 3 години стаж има
позиция, а един със 7 години няма.

> **Въпрос.** Има ли граница, която познава и 14-те преподаватели?

<details><summary>Отговор</summary>

Няма. Трима преподаватели имат по 3 години стаж, но само един от тях има
позиция. Щом стажът им е еднакъв, всяко правило по стажа им дава един и
същ клас. Данни, които никоя граница не разделя без грешка, се наричат
**линейно неразделими**.

</details>

Сега и двата признака. Всеки преподавател е точка в равнината: стажът е по
хоризонталата, а публикациите по вертикалата. Затова границата вече е
**права**. Една такава права е

`5 · стаж + 2 · публикации = 49`

За точките над правата 5 · стаж + 2 · публикации е повече от 49 и
правилото казва "позиция". За точките под нея казва "без позиция".
Следващата клетка проверява колко преподаватели познава това правило и
какво става с двете изключения.

In [ ]:
def scatter_tenure(ax):
    """Преподавателите по стаж и публикации, оцветени по клас."""
    for label, color, name in [(1, "#2a78d6", "с позиция (1)"), (0, "#eb6834", "без позиция (0)")]:
        mask = tenure["tenured"] == label
        ax.scatter(tenure.loc[mask, "years"], tenure.loc[mask, "publications"], s=60, color=color,
                   zorder=3, label=name)
    ax.set(xlim=(0, 13), ylim=(0, 20), xlabel="години стаж", ylabel="брой публикации")


score_2d = 5 * tenure["years"] + 2 * tenure["publications"] - 49
rule_2d = (score_2d > 0).astype(int)
print("Правилни според правата:", int((rule_2d == tenure["tenured"]).sum()), "от", len(tenure))
display(tenure.assign(**{"5 · стаж + 2 · публикации - 49": score_2d})[rule != tenure["tenured"]])

years_line = np.array([0, 13])
fig, ax = plt.subplots(figsize=(7, 4.8))
scatter_tenure(ax)
ax.plot(years_line, (49 - 5 * years_line) / 2, color="black", linestyle="--", linewidth=1,
        label="5 · стаж + 2 · публикации = 49")
ax.set(title="Постоянна позиция според стажа и публикациите")
ax.legend(loc="upper right")
plt.show()

Правата познава и 14-те преподаватели. Двете изключения вече имат
обяснение: преподавателят с 3 години стаж и позиция има 18 публикации,
най-много от всички, а този със 7 години стаж без позиция има само 3.

Само по стажа данните са линейно неразделими, а по стажа и публикациите
са **линейно разделими**: има права, която разделя класовете без нито
една грешка. Как се намира такава права, показва перцептронът в
[част 2](#chast-2).

<a id="chast-2"></a>
## Част 2. Перцептрон

<a id="t3"></a>
### 3. Знакът на линеен резултат

Най-простият начин линеен модел да избира клас е **перцептронът**,
използван вече в упражнение 2, части 3 и 4. Той смята същата сума като
линейната регресия, wᵀx + b, но гледа само знака ѝ. Класовете са +1 и -1:

`ŷ = sign(wᵀx + b)`

Ако wᵀx + b > 0, предсказанието е +1, а иначе е -1. Границата на решението
е там, където wᵀx + b = 0. Векторът w е перпендикулярен на границата и
сочи към положителния клас.

Примерът са осем точки x1 до x8 с два признака f1 и f2. За простота b = 0,
а текущото тегло е w = (1, 0). Тогава границата е вертикалната ос.

In [ ]:
points = pd.DataFrame({
    "f1": [-1, 1, 2, 2, -1, -2, -2, 1],
    "f2": [2, 1, 2, 1, -1, -1, -2, -2],
    "y":  [1, 1, 1, 1, -1, -1, -1, -1],
}, index=[f"x{i}" for i in range(1, 9)])
w = np.array([1.0, 0.0])
points["wᵀx"] = points[["f1", "f2"]].to_numpy() @ w
points["ŷ"] = np.sign(points["wᵀx"]).astype(int)
display(points.T)
print("Грешно класифицирани:", list(points.index[points["ŷ"] != points["y"]]))

<a id="t4"></a>
### 4. Правилото за обучение

Перцептронът се учи от грешките си. Когато познае, не прави нищо. Когато
сгреши за пример x с истински клас y (+1 или -1), поправя теглото така:

`w = w + η · y · x`

η > 0 е **скоростта на обучение** (learning rate): колко голяма е една
поправка. Знакът на y казва накъде е поправката:

- пропуснат положителен пример (y = +1, а ŷ = -1) се добавя към w и w се
  завърта към него;
- погрешно приет отрицателен пример (y = -1, а ŷ = +1) се изважда от w и w
  се завърта далеч от него.

Ако класовете са 1 и 0, същото правило се записва като
`w = w + η · (y - ŷ) · x`. Тогава не е нужно да се проверява дали има
грешка: при верен отговор y - ŷ = 0 и нищо не се променя, а при грешка
y - ŷ е 1 или -1 и пак показва посоката.

Тук грешат x1 (положителен) и x8 (отрицателен). Двете поправки се правят
наведнъж, с η = 0.5: w = w + η · x1 - η · x8.

In [ ]:
eta = 0.5
x1 = points.loc["x1", ["f1", "f2"]].to_numpy(dtype=float)
x8 = points.loc["x8", ["f1", "f2"]].to_numpy(dtype=float)
w_new = w + eta * x1 - eta * x8
print("w преди:", w, "| w след:", w_new)

scores_new = points[["f1", "f2"]].to_numpy() @ w_new
print("wᵀx след:", scores_new)
print("Грешно класифицирани след:", int((np.sign(scores_new) != points["y"]).sum()))

fig, axes = plt.subplots(1, 2, figsize=(10, 4.6), sharey=True)
for ax, weights, title in zip(axes, [w, w_new], ["(а) текущо тегло w", "(б) ново тегло w"]):
    ax.scatter(points["f1"], points["f2"], s=80, zorder=3,
               c=np.where(points["y"] == 1, "#2a78d6", "#eb6834"))
    for name, row in points.iterrows():
        ax.annotate(name, (row["f1"], row["f2"]), xytext=(6, 4), textcoords="offset points")
    # границата wᵀx = 0 минава през (0, 0) и е перпендикулярна на w
    ax.axline((0, 0), (-weights[1], weights[0]), color="black", linewidth=1)
    ax.annotate("", xy=weights, xytext=(0, 0), arrowprops={"arrowstyle": "->", "color": "black", "lw": 1.5})
    ax.text(weights[0] + 0.15, weights[1] + 0.15, "w")
    ax.set(xlim=(-3, 3), ylim=(-3, 3), xlabel="f1", title=title)
    ax.set_aspect("equal")
axes[0].set_ylabel("f2")
plt.show()

След една стъпка w = (0, 2) и границата е хоризонталната ос. Всички осем
точки са от правилната страна: положителните са горе, а отрицателните
долу. Новото тегло е завъртяно към x1 и далеч от x8.

`Perceptron` от scikit-learn прави същото много пъти: минава през тренировъчните примери отново и отново и поправя теглата при всяка грешка. Едно минаване през всички примери се нарича **епоха**. Етикетите могат да са 0 и 1: scikit-learn сам ги превръща в -1 и +1.

<a id="t5"></a>
### 5. Кога перцептронът спира

Има два случая:

- ако някоя права разделя класовете без грешка (данните са **линейно
  разделими**), перцептронът със сигурност намира такава права и спира да
  променя теглата;
- ако такава права няма, винаги остава грешка, всяка грешка мести теглата
  и перцептронът не спира.

Преподавателите показват и двата случая: само по стажа не са разделими, а
по стажа и публикациите са разделими
([точка 2 "Граница на решението"](#t2)). Следващата клетка обучава
`Perceptron` с 50, 100 и 200 епохи. `tol=None` е важно: без него
scikit-learn спира, ако броят на грешките не намалява 5 епохи подред, дори
когато още има грешки.

In [ ]:
from sklearn.linear_model import Perceptron

two_features = ["years", "publications"]
for features in [["years"], two_features]:
    print("Признаци:", ", ".join(features))
    for epochs in [50, 100, 200]:
        perceptron = Perceptron(tol=None, max_iter=epochs, random_state=42)
        perceptron.fit(tenure[features], tenure["tenured"])
        errors = int((perceptron.predict(tenure[features]) != tenure["tenured"]).sum())
        print(f"  {epochs:3} епохи: грешки {errors}, w = {perceptron.coef_[0]}, b = {perceptron.intercept_[0]}")

# последният обучен перцептрон е с двата признака и 200 епохи
w_years, w_pubs = perceptron.coef_[0]
b_perceptron = perceptron.intercept_[0]
fig, ax = plt.subplots(figsize=(7, 4.8))
scatter_tenure(ax)
ax.plot(years_line, -(b_perceptron + w_years * years_line) / w_pubs, color="black", linewidth=1.2,
        label=f"перцептрон: {w_years:.0f} · стаж + {w_pubs:.0f} · публикации = {-b_perceptron:.0f}")
ax.plot(years_line, (49 - 5 * years_line) / 2, color="gray", linestyle="--", linewidth=1,
        label="5 · стаж + 2 · публикации = 49")
ax.set(title="Границата, която намира перцептронът")
ax.legend(loc="upper right")
plt.show()

Със стажа и публикациите перцептронът намира w = (8, 3) и b = -77 и не
греши. При 50, 100 и 200 епохи теглата са едни и същи: щом няма грешка,
няма и поправка. Правата 8 · стаж + 3 · публикации = 77 е различна от
правата в [точка 2 "Граница на решението"](#t2), но и тя разделя
преподавателите без грешка. Разделящите прави са много и перцептронът
спира на първата, която намери.

Само със стажа грешки остават, а теглата са различни при всеки брой епохи:
перцептронът не спира. Логистичната регресия в [част 3](#chast-3) дава
една най-добра граница и когато класовете не могат да се разделят.

In [ ]:
# --- ДА ОПИТАМЕ 1 ---
#
# Проследете броя на грешките по епохи. partial_fit прави само една епоха,
# затова го извикайте 40 пъти и след всяка епоха пребройте грешно
# класифицираните преподаватели. Сравнете обучението само по стажа с
# обучението по стажа и публикациите. При първото извикване partial_fit
# иска classes=[0, 1]. Защо при двата признака грешките остават 0, щом
# веднъж станат 0? Защо само по стажа никога не стават 0?

solution.to_test_1(tenure)

<a id="chast-3"></a>
## Част 3. Логистична регресия

<a id="t6"></a>
### 6. Защо не линейна регресия

Класът може да се разглежда като числото 0 или 1. Тогава може да се обучи
линейна регресия, както в упражнение 3. Проблемът е, че правата продължава
нагоре и надолу без край: прогнозата може да е под 0 или над 1, а класът е
само 0 или 1. Следващата клетка обучава `LinearRegression` върху стажа.

In [ ]:
from sklearn.linear_model import LinearRegression

line = LinearRegression().fit(tenure[["years"]], tenure["tenured"])
check = pd.DataFrame({"years": [0, 1, 6, 12, 20]})
check["прогноза на правата"] = line.predict(check[["years"]]).round(3)
display(check.T)
print("Правата стига 0.5 при", round((0.5 - line.intercept_) / line.coef_[0], 2), "години")

При 0 и 1 година прогнозата е отрицателна, а при 12 и 20 години е над 1.
Такова число не може да е вероятност. Правата все пак дава граница там,
където стига 0.5, т.е. при около 6.1 години. Тази граница обаче се мести и
от примери, които са далеч от нея (ДА ОПИТАМЕ 2).

<a id="t7"></a>
### 7. Сигмоидната функция

Логистичната регресия смята същата сума z = wᵀx + b, но после я прекарва
през **сигмоидната** (логистичната) функция:

`σ(z) = 1 / (1 + e⁻ᶻ)`

Сигмоидата превръща всяко число в число между 0 и 1, т.е. във вероятност:

- при голямо положително z стойността е почти 1, а при голямо отрицателно
  z е почти 0;
- σ(0) = 0.5 и σ(-z) = 1 - σ(z);
- около нулата се променя бързо, а далеч от нея почти не се променя.

Производната на сигмоидата има прост вид: σ'(z) = σ(z) · (1 - σ(z)).
Следващата клетка проверява и двете неща с числа.

In [ ]:
def sigmoid(z):
    """Сигмоидната функция 1 / (1 + e⁻ᶻ)."""
    return 1 / (1 + np.exp(-z))


z_values = np.array([-6, -2, -1, 0, 1, 2, 6])
display(pd.DataFrame({"z": z_values, "σ(z)": sigmoid(z_values).round(3)}).T)

# наклонът между две близки точки срещу σ(z) · (1 - σ(z))
z, step = 1.0, 0.001
print("Наклон около z = 1: ", round((sigmoid(z + step) - sigmoid(z - step)) / (2 * step), 4))
print("σ(1) · (1 - σ(1)):  ", round(sigmoid(z) * (1 - sigmoid(z)), 4))

grid = np.linspace(-6, 6, 200)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(grid, sigmoid(grid), color="#2a78d6", label="σ(z)")
ax.plot(grid, sigmoid(grid) * (1 - sigmoid(grid)), color="#eb6834", label="σ'(z) = σ(z) · (1 - σ(z))")
ax.axhline(0.5, color="gray", linewidth=0.8, linestyle=":")
ax.set(xlabel="z", title="Сигмоидната функция и производната ѝ")
ax.legend()
plt.show()

Производната е най-голяма при z = 0, където е 0.25, и е почти 0 далеч от
нулата. Формулата σ' = σ · (1 - σ) е удобна, защото производната се смята
от самата стойност на σ. Тя се използва в
[точка 13 "Градиентно изкачване"](#t13).

<a id="t8"></a>
### 8. Логистична регресия със scikit-learn

За стажа моделът е

`h(x) = P(y = 1 | x) = σ(b + w · years)`

Той връща вероятността за клас 1, т.е. за постоянна позиция. За да се
получи клас, вероятността се сравнява с **прага** 0.5: ŷ = 1 при p ≥ 0.5 и
ŷ = 0 при p < 0.5. Вероятността е поне 0.5 точно когато
z = b + w · years ≥ 0. Затова границата отново е z = 0, както при
перцептрона, т.е. years = -b / w.

`LogisticRegression` по подразбиране добавя **регуляризация**. Тя свива теглата към нула и ще бъде обяснена по-нататък в курса. С `C=np.inf` регуляризацията се изключва и теглата се избират само по метода от [част 5](#chast-5). `predict_proba` връща по една колона за всеки клас: P(y = 0 | x) и P(y = 1 | x).

In [ ]:
from sklearn.linear_model import LogisticRegression

logit_model = LogisticRegression(C=np.inf).fit(tenure[["years"]], tenure["tenured"])
b, w = logit_model.intercept_[0], logit_model.coef_[0, 0]
print(f"b = {b:.3f} | w = {w:.3f} | граница: {-b / w:.2f} години")

tenure_pred = tenure.assign(**{
    "P(1)": logit_model.predict_proba(tenure[["years"]])[:, 1].round(3),
    "ŷ": logit_model.predict(tenure[["years"]]),
})
display(tenure_pred.T)

grid = pd.DataFrame({"years": np.linspace(0, 14, 200)})
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(grid["years"], logit_model.predict_proba(grid)[:, 1], color="#2a78d6", label="P(tenured = 1)")
ax.plot(grid["years"], line.predict(grid), color="#eb6834", linestyle="--", label="правата от точка 6")
ax.scatter(tenure["years"], stacked(tenure), s=50, zorder=3,
           c=np.where(tenure["tenured"] == 1, "#2a78d6", "#eb6834"))
ax.axvline(-b / w, color="black", linewidth=0.8, linestyle=":")
ax.axhline(0.5, color="gray", linewidth=0.8, linestyle=":")
ax.set(xlabel="години стаж", ylabel="вероятност", title="Логистичната регресия за стажа")
ax.legend(loc="upper left")
plt.show()

b ≈ -4.04 и w ≈ 0.68. Вероятността минава 0.5 при 5.95 години, почти
колкото правилото "поне 6 години". Моделът греши за същите двама
преподаватели, но казва и **колко е сигурен**: вероятността за позиция е
0.034 при 1 година, 0.509 при 6 години и 0.984 при 12 години. Кривата
остава между 0 и 1, а правата излиза извън тях.

In [ ]:
# --- ДА ОПИТАМЕ 2 ---
#
# Добавете двама преподаватели с 30 и 35 години стаж и постоянна позиция.
# За тях няма никакво съмнение. Обучете отново правата и логистичната
# регресия. Колко се мести всяка граница: мястото, където правата стига
# 0.5, и -b / w? Защо далечните примери местят правата?

solution.to_test_2(tenure)

<a id="chast-4"></a>
## Част 4. Шансове и logit

Шансовете и logit обясняват какво означава моделът и неговите тегла.

<a id="t9"></a>
### 9. Шансове

**Шансовете** (odds) за клас 1 са вероятността за клас 1, разделена на
вероятността за клас 0:

`odds = p / (1 - p)`

При p = 0.8 шансовете са 0.8 / 0.2 = 4, т.е. "4 към 1": клас 1 е четири
пъти по-вероятен от клас 0. При p = 0.5 шансовете са 1 към 1. Шансовете са
между 0 и безкрайност. **Logit** е логаритъмът на шансовете и може да е
всяко реално число:

`logit(p) = ln(p / (1 - p))`

Сигмоидата е обратната функция на logit: ако z = logit(p), то p = σ(z).

In [ ]:
p = np.array([0.1, 0.2, 0.5, 0.8, 0.9])
odds = p / (1 - p)
table = pd.DataFrame({"p": p, "шансове": odds.round(3), "logit(p)": np.log(odds).round(3),
                      "σ(logit(p))": sigmoid(np.log(odds)).round(3)})
display(table)

Вероятностите 0.2 и 0.8 дават шансове 0.25 и 4 и logit -1.386 и 1.386:
logit е симетричен около p = 0.5. Последната колона връща вероятностите,
защото сигмоидата е обратен на logit.

<a id="t10"></a>
### 10. Логаритъмът на шансовете е линеен

Логистичната регресия може да се запише и така:

`ln(p(x) / (1 - p(x))) = b + wᵀx`

Т.е. логаритъмът на шансовете е линейна сума, както прогнозата в линейната
регресия. Ако равенството се реши за p, се получава отново p = σ(b + wᵀx).
`decision_function` връща тази сума z = b + wᵀx.

Оттук идва смисълът на теглото:

- още една година стаж добавя w към логаритъма на шансовете;
- затова шансовете се умножават по eʷ;
- вероятността **не** расте с еднаква стъпка: тя се променя най-много
  около границата, където кривата е най-стръмна.

При линейната регресия теглото е промяната на прогнозата, а тук е
промяната на логаритъма на шансовете.

In [ ]:
compare = pd.DataFrame({"years": [5, 6, 11, 12]})
p_compare = logit_model.predict_proba(compare[["years"]])[:, 1]
odds_compare = p_compare / (1 - p_compare)
compare["P(1)"] = p_compare.round(3)
compare["шансове"] = odds_compare.round(3)
compare["logit"] = np.log(odds_compare).round(3)
compare["z = b + w · years"] = logit_model.decision_function(compare[["years"]]).round(3)
display(compare)
print("eʷ =", round(np.exp(w), 3))
print("Шансовете при 6 години, разделени на тези при 5 години:", round(odds_compare[1] / odds_compare[0], 3))

logit на вероятността съвпада със z: логаритъмът на шансовете е точно
линейният резултат. Всяка година умножава шансовете по eʷ ≈ 1.97, т.е.
почти ги удвоява. Вероятността обаче расте с 0.165 между 5 и 6 години и
само с 0.015 между 11 и 12 години.

In [ ]:
# --- ДА ОПИТАМЕ 3 ---
#
# След колко години стаж вероятността за постоянна позиция става 0.9? А 0.1?
# Решете b + w · years = logit(p) спрямо years, вместо да търсите с проби.
# Проверете отговора с predict_proba. На какво разстояние от границата -b / w
# са двата отговора и защо?

solution.to_test_3(logit_model)

<a id="chast-5"></a>
## Част 5. Максимално правдоподобие

<a id="t11"></a>
### 11. Правдоподобие

При линейната регресия теглата се избираха така, че RSS да е най-малка.
При логистичната регресия се използва друга идея, **методът на
максималното правдоподобие** (maximum likelihood estimation, MLE): добрите
тегла дават голяма вероятност на това, което наистина се е случило.

За всеки пример моделът дава вероятност на наблюдавания клас:

- ако y = 1, тя е h(x);
- ако y = 0, тя е 1 - h(x);
- двата случая се записват заедно като h(x)ʸ · (1 - h(x))¹⁻ʸ: при y = 1
  остава първият множител, а при y = 0 вторият.

**Правдоподобието** L е вероятността моделът да даде точно наблюдаваните
класове на всички примери заедно. Ако примерите са **независими**, т.е.
класът на един преподавател не зависи от класа на друг, L е произведение
от вероятностите за отделните примери. Например, ако моделът дава
вероятности 0.9, 0.8 и 0.7 на наблюдаваните класове на три примера,
L = 0.9 · 0.8 · 0.7 = 0.504.

Без независимост всеки множител може да зависи и от предишните примери.
Това е **верижното правило**: P(A, B, C) = P(A) · P(B | A) · P(C | A, B).
При независими примери P(B | A) = P(B) и P(C | A, B) = P(C) и остава
обикновено произведение.

Произведението на много числа под 1 бързо става много малко. Затова се
използва **логаритъмът на правдоподобието**. Логаритъмът на произведение е
сума от логаритми:

`l(w) = Σ [y⁽ⁱ⁾ · log h(x⁽ⁱ⁾) + (1 - y⁽ⁱ⁾) · log(1 - h(x⁽ⁱ⁾))]`

Следващата клетка смята примера с трите вероятности и сравнява теглата на
модела за стажа с два други избора на b и w.

In [ ]:
three_examples = np.array([0.9, 0.8, 0.7])
print(f"Три примера: L = {three_examples.prod():.3f}, l = ln L = {np.log(three_examples).sum():.3f}")
print()


def observed_probability(b, w):
    """Вероятността на наблюдавания клас за всеки преподавател."""
    h = sigmoid(b + w * tenure["years"])
    return np.where(tenure["tenured"] == 1, h, 1 - h)


candidates = {"модела": (b, w), "по-полегата крива": (-3.0, 0.5), "почти стъпало при 6": (-18.0, 3.0)}
for name, (b_try, w_try) in candidates.items():
    observed = observed_probability(b_try, w_try)
    print(f"{name:20} b = {b_try:6.2f}, w = {w_try:4.2f} | L = {observed.prod():.2e}"
          f" | l = {np.log(observed).sum():.3f}")

display(pd.DataFrame({
    "years": tenure["years"], "tenured": tenure["tenured"],
    "модела": observed_probability(b, w).round(3),
    "почти стъпало": observed_probability(-18.0, 3.0).round(3),
}).T)

Таблицата показва вероятността на наблюдавания клас. Моделът има
най-голямо правдоподобие от трите. Почти стъпалото при 6 години е много
уверено и затова плаща скъпо за двете грешки: на преподавателя с 3 години
и позиция дава вероятност за позиция 0.0001. Едно такова число е
достатъчно, за да направи цялото произведение малко.

<a id="t12"></a>
### 12. Log loss

Най-голямото l(w) и най-малкото -l(w) се получават при едни и същи тегла.
Обикновено -l(w) се дели на броя на примерите m. Така се получава средното
на минус логаритъма от вероятността на наблюдавания клас:

`log loss = -1/m · Σ log p(y⁽ⁱ⁾ | x⁽ⁱ⁾) = -l(w) / m`

В scikit-learn това е функцията `log_loss`. `LogisticRegression(C=np.inf)`
намира теглата с най-малък log loss, т.е. с най-голямо правдоподобие.

In [ ]:
from sklearn.metrics import log_loss

p_model = logit_model.predict_proba(tenure[["years"]])[:, 1]
print("l(w) на модела:            ", round(np.log(observed_probability(b, w)).sum(), 3))
print("log_loss · брой примери:   ", round(log_loss(tenure["tenured"], p_model) * len(tenure), 3))

In [ ]:
# --- ДА ОПИТАМЕ 4 ---
#
# Може ли друго w да даде по-голямо правдоподобие? Оставете b от модела и
# изчислете l(w) за 300 стойности на w между 0 и 2. Начертайте кривата и
# отбележете w на модела. Къде е най-високата точка? Сравнете с ДА ОПИТАМЕ 1
# от упражнение 3, част 2, където RSS имаше най-ниска точка.

solution.to_test_4(tenure, logit_model)

<a id="t13"></a>
### 13. Градиентно изкачване

При линейната регресия има формула за най-добрите тегла. При логистичната
регресия такава формула няма. Затова върхът на l(w) се търси с малки
стъпки, като при изкачване на хълм.

**Градиентът** ∇l(w) е векторът от частните производни на l по всяко
тегло. Той сочи посоката, в която l расте най-бързо. **Градиентното
изкачване** прави малка стъпка в тази посока:

`w = w + α · ∇l(w)`

Знакът е плюс, защото се търси най-голямото l. Когато се търси най-малка
стойност, например на RSS, стъпката е в обратната посока и методът се
нарича градиентно спускане.

За един пример (x, y) производната се смята с формулата σ' = σ · (1 - σ)
от [точка 7 "Сигмоидната функция"](#t7) и излиза много проста:

`∂l/∂wⱼ = (y - h(x)) · xⱼ`, откъдето `wⱼ = wⱼ + α · (y - h(x)) · xⱼ`

Ако стъпката се прави след всеки отделен пример, методът е **стохастично**
градиентно изкачване. Свободният член b се обновява по същия начин, като
за него xⱼ = 1.

**Сравнение с LMS.** При линейната регресия същата идея, приложена към
квадрата на грешката (y - wᵀx)² / 2, дава правилото LMS (least mean
squares): wⱼ = wⱼ + α · (y - ŷ) · xⱼ, където ŷ = wᵀx. Двете правила
изглеждат еднакво, но не са: при логистичната регресия h(x) = σ(wᵀx) не е
права. Перцептронът също има този вид, с ŷ = sign(wᵀx).

Следващата клетка прави една стъпка на ръка за преподавателя с 8 години
стаж и позиция. Тя тръгва от b = 0 и w = 0, с α = 0.01. След това
`SGDClassifier` от scikit-learn прави същите стъпки много пъти:
`loss="log_loss"` е логистичната регресия, `penalty=None` изключва
регуляризацията, а `learning_rate="constant"` и `eta0=0.01` задават α.

In [ ]:
from sklearn.linear_model import SGDClassifier

alpha, b_step, w_step = 0.01, 0.0, 0.0
years_i, y_i = 8, 1
h = sigmoid(b_step + w_step * years_i)
b_step = b_step + alpha * (y_i - h) * 1
w_step = w_step + alpha * (y_i - h) * years_i
print(f"h = {h} | след стъпката: b = {b_step:.3f}, w = {w_step:.3f}")

sgd_model = SGDClassifier(loss="log_loss", penalty=None, learning_rate="constant", eta0=0.01,
                          max_iter=2000, tol=None, random_state=42)
sgd_model.fit(tenure[["years"]], tenure["tenured"])
print(f"SGDClassifier:      b = {sgd_model.intercept_[0]:.3f}, w = {sgd_model.coef_[0, 0]:.3f}")
print(f"LogisticRegression: b = {b:.3f}, w = {w:.3f}")

В началото h = 0.5 за всеки пример. Преподавателят е в клас 1, затова
y - h = 0.5 > 0 и двете тегла растат. Ако беше в клас 0, щяха да намалеят.
Стъпката е по-голяма, когато моделът греши повече (|y - h| е голямо) и
когато признакът е голям.

След 2000 епохи `SGDClassifier` стига почти до теглата на
`LogisticRegression`. `LogisticRegression` използва по-бърз метод
(lbfgs), но търси същия максимум.

In [ ]:
# --- ДА ОПИТАМЕ 5 ---
#
# По стажа и публикациите преподавателите са линейно разделими (точка 2).
# Обучете LogisticRegression с двата признака, с C=np.inf и с C=1 (по
# подразбиране). Какви са теглата и вероятностите за двете изключения
# (3 години с позиция и 7 години без позиция)? Изчислете l(w) за правата
# 5 · стаж + 2 · публикации - 49, като умножите теглата и свободния член по
# s = 0.1, 0.2, 0.5, 1, 2 и 5. Границата не се мени, а кривата става
# по-стръмна. Има ли най-голямо правдоподобие? Какво прави регуляризацията?

solution.to_test_5(tenure)

<a id="chast-6"></a>
## Част 6. Вероятност при два признака

<a id="t14"></a>
### 14. Линии с еднаква вероятност

При два признака границата z = b + w₁x₁ + w₂x₂ = 0 е права
([точка 2 "Граница на решението"](#t2)). Логистичната регресия дава и
вероятност за всяка точка от равнината.

Примерът тук е с ирисите, а не с преподавателите. Преподавателите по стаж
и публикации се разделят без грешка и тогава теглата на логистичната
регресия растат без край (ДА ОПИТАМЕ 5). При ирисите видовете се
застъпват. Въпросът е дали цветето е virginica според дължината и ширината
на венчелистчето. Цветята се разделят на тренировъчни и тестови, а моделът
се сравнява с постоянна прогноза (`DummyClassifier`), която винаги
отговаря с най-честия клас.

По цялата права z = const вероятността е една и съща. Затова линиите с
вероятност 0.1 и 0.9 са успоредни на границата: на тях
z = logit(0.1) = -2.197 и z = logit(0.9) = 2.197.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split

iris = load_iris(as_frame=True)
flowers = iris.frame.rename(columns={
    "sepal length (cm)": "sepal_length", "sepal width (cm)": "sepal_width",
    "petal length (cm)": "petal_length", "petal width (cm)": "petal_width",
})
flowers["species"] = iris.target_names[flowers["target"]]

petal = ["petal_length", "petal_width"]
is_virginica = (flowers["species"] == "virginica").astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    flowers[petal], is_virginica, test_size=0.25, random_state=42, stratify=is_virginica
)
virginica_model = LogisticRegression(C=np.inf).fit(X_train, y_train)
baseline = DummyClassifier().fit(X_train, y_train)
b_v = virginica_model.intercept_[0]
w_length, w_width = virginica_model.coef_[0]
print(f"b = {b_v:.2f} | w = ({w_length:.2f}, {w_width:.2f})")
print("Тестова точност: модел", round(virginica_model.score(X_test, y_test), 3),
      "| постоянна прогноза", round(baseline.score(X_test, y_test), 3))

lengths = np.array([1.0, 7.0])
fig, ax = plt.subplots(figsize=(7, 5))
for label, color, name in [(1, "#2a78d6", "virginica"), (0, "#eb6834", "другите видове")]:
    mask = y_train == label
    ax.scatter(X_train.loc[mask, "petal_length"], X_train.loc[mask, "petal_width"], s=25, color=color, label=name)
for level, style in [(0.1, ":"), (0.5, "-"), (0.9, "--")]:
    z_level = np.log(level / (1 - level))
    # b + w₁ · length + w₂ · width = z_level, решено спрямо width
    ax.plot(lengths, (z_level - b_v - w_length * lengths) / w_width, color="black", linestyle=style,
            label=f"P(virginica) = {level}")
ax.set(xlim=(0.8, 7.2), ylim=(0, 2.7), xlabel="petal_length (cm)", ylabel="petal_width (cm)",
       title="Virginica срещу другите видове")
ax.legend()
plt.show()

Моделът познава 94.7% от тестовите цветя, а постоянната прогноза 65.8%.
Границата е права. Между линиите с 0.1 и 0.9 моделът не е сигурен: там
versicolor и virginica се застъпват. Далеч от границата вероятността е
почти 0 или 1.

<a id="chast-7"></a>
## Част 7. Многокласова класификация

<a id="t15"></a>
### 15. Едно срещу всички

Когато класовете са K, подходът **едно срещу всички** (one-vs-rest, OvR)
обучава K отделни двоични класификатора. За ирисите те са три: setosa
срещу другите, versicolor срещу другите и virginica срещу другите. Всеки
дава резултат zₖ = wₖᵀx + bₖ: колкото е по-голям, толкова повече примерът
прилича на клас k. Резултатите zₖ не са вероятности.

Всеки двоичен класификатор има своя област "да" (zₖ > 0). Тези области
могат да се припокриват, а някоя точка може да не е в нито една от тях.
Затова решението не е "кой класификатор казва да", а сравнение: точката
получава класа с най-голям резултат, ŷ = argmaxₖ zₖ.

Признаците са дължината и ширината на чашелистчето, стандартизирани със
`StandardScaler` (упражнение 2, част 2). `OneVsRestClassifier` обучава по
един `LogisticRegression` за всеки клас.

In [ ]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import StandardScaler

sepal = ["sepal_length", "sepal_width"]
sepal_train, sepal_test, species_train, species_test = train_test_split(
    flowers[sepal], flowers["species"], test_size=0.25, random_state=42, stratify=flowers["species"]
)
scaler = StandardScaler().fit(sepal_train)
sepal_train_std, sepal_test_std = scaler.transform(sepal_train), scaler.transform(sepal_test)

ovr_model = OneVsRestClassifier(LogisticRegression(C=np.inf)).fit(sepal_train_std, species_train)
scores = ovr_model.decision_function(sepal_test_std[:5])
display(pd.DataFrame(scores.round(2), columns=ovr_model.classes_).assign(
    argmax=ovr_model.classes_[scores.argmax(axis=1)], predict=ovr_model.predict(sepal_test_std[:5])))
print("Тестова точност OvR:", round(ovr_model.score(sepal_test_std, species_test), 3))

In [ ]:
SPECIES_COLORS = ["#2a78d6", "#eb6834", "#4d4d4d"]


def plot_regions(model, X, labels, title):
    """Цветните области на предсказания клас и цветята върху тях."""
    xx, yy = np.meshgrid(np.linspace(-2.6, 3.1, 300), np.linspace(-2.6, 3.1, 300))
    predicted = model.predict(np.c_[xx.ravel(), yy.ravel()])
    codes = pd.Categorical(predicted, categories=model.classes_).codes.reshape(xx.shape)
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    ax.contourf(xx, yy, codes, levels=[-0.5, 0.5, 1.5, 2.5], colors=SPECIES_COLORS, alpha=0.2)
    for color, species in zip(SPECIES_COLORS, model.classes_):
        mask = labels == species
        ax.scatter(X[mask, 0], X[mask, 1], s=22, color=color, edgecolor="white", linewidth=0.5, label=species)
    ax.set(xlim=(-2.6, 3.1), ylim=(-2.6, 3.1), xlabel="sepal_length (стандартизирана)",
           ylabel="sepal_width (стандартизирана)", title=title)
    return ax


ax = plot_regions(ovr_model, sepal_train_std, species_train.to_numpy(), "Едно срещу всички")
xs = np.array([-2.6, 3.1])
for color, estimator in zip(SPECIES_COLORS, ovr_model.estimators_):
    b_k = estimator.intercept_[0]
    w_k1, w_k2 = estimator.coef_[0]
    # границата на двоичния класификатор: b_k + w_k1 · x1 + w_k2 · x2 = 0
    ax.plot(xs, -(b_k + w_k1 * xs) / w_k2, color=color, linestyle="--", linewidth=1.5)
ax.legend()
plt.show()

Във втория и третия ред на таблицата и трите резултата са отрицателни:
цветето не е в нито една от трите области "да", но пак получава класа с
най-голям резултат.

Всяка пунктирна права е границата на един двоичен класификатор. Цветните
области показват кой от трите резултата е най-голям. Границите между
областите са части от прави, защото там два резултата са равни. Versicolor
и virginica се застъпват силно по чашелистчето, затова точността е около
76%.

<a id="t16"></a>
### 16. Softmax

Резултатите на OvR не са вероятности. **Softmax** превръща K резултата във
вероятности:

`P(Y = k | x) = e^(zₖ) / Σⱼ e^(zⱼ)`

Всяка вероятност е между 0 и 1, а сборът им е 1. Предсказанието е класът с
най-голяма вероятност. Пример: z = (2, 1, 0).

In [ ]:
def softmax(z):
    """e^(zₖ) / Σⱼ e^(zⱼ) за всеки клас; работи и за таблица от резултати."""
    exp_z = np.exp(z)
    return exp_z / exp_z.sum(axis=-1, keepdims=True)


z = np.array([2.0, 1.0, 0.0])
print("e^z:", np.exp(z).round(3), "| сбор:", round(np.exp(z).sum(), 3))
print("softmax(z):", softmax(z).round(3), "| сбор:", round(softmax(z).sum(), 3))
print("ŷ = клас", softmax(z).argmax() + 1)

softmax(z) ≈ (0.665, 0.245, 0.090) и предсказанието е клас 1. Разликата
между z₁ и z₂ е 1, затова P₁ / P₂ = e ≈ 2.7. При два класа softmax е
сигмоидата: e^(z₁) / (e^(z₁) + e^(z₂)) = σ(z₁ - z₂).

<a id="t17"></a>
### 17. Многокласова логистична регресия

При повече от два класа `LogisticRegression` използва softmax. Тя обучава
наведнъж K резултата, по един за всеки клас, и търси най-голямо
правдоподобие за вероятностите от softmax. Границата между класове k и ℓ е
там, където zₖ = zₗ, т.е. (wₖ - wₗ)ᵀx + (bₖ - bₗ) = 0. Това е права,
затова и тук границите са линейни.

In [ ]:
softmax_model = LogisticRegression(C=np.inf).fit(sepal_train_std, species_train)
proba = softmax_model.predict_proba(sepal_test_std[:5])
z_test = softmax_model.decision_function(sepal_test_std[:5])
display(pd.DataFrame(proba.round(3), columns=softmax_model.classes_).assign(сбор=proba.sum(axis=1).round(3)))
print("softmax на decision_function съвпада с predict_proba:", np.allclose(softmax(z_test), proba))
print("Тестова точност: softmax", round(softmax_model.score(sepal_test_std, species_test), 3),
      "| OvR", round(ovr_model.score(sepal_test_std, species_test), 3))

ax = plot_regions(softmax_model, sepal_train_std, species_train.to_numpy(), "Многокласова логистична регресия")
ax.legend()
plt.show()

Сборът на вероятностите за всяко цвете е 1, а softmax на
`decision_function` дава точно `predict_proba`. Границите между областите
отново са прави. Точността е като при OvR, но сега за всяко цвете има и
вероятности за трите вида.

In [ ]:
# --- ДА ОПИТАМЕ 6 ---
#
# Вземете z на второто тестово цвете от softmax_model.decision_function.
# Моделът се колебае за него между versicolor и virginica.
# 1) Увеличете z на versicolor с 1, без да променяте другите. Как се
#    променят трите вероятности?
# 2) Добавете едно и също число, например 5, към трите z. Защо
#    вероятностите не се променят?

solution.to_test_6(softmax_model, sepal_test_std)

<a id="obobshtenie"></a>
## Обобщение

1. Класификацията предсказва клас от краен набор. Класът 1 е положителен,
   а 0 отрицателен. Границата на решението е точка при един признак и
   права при два. Само по стажа преподавателите са линейно неразделими, а
   по стажа и публикациите са разделими.
2. Перцептронът предсказва sign(wᵀx + b) и поправя теглата само при грешка:
   w = w + η · y · x при етикети +1 и -1 (или w = w + η · (y - ŷ) · x при
   0 и 1). При линейно разделими данни намира граница без грешки, а при
   неразделими теглата не спират да се променят.
3. Линейната регресия върху 0 и 1 дава прогнози извън [0, 1] и граница,
   която зависи от далечни примери. Логистичната регресия прекарва
   z = wᵀx + b през сигмоидата и връща вероятност. Класът е 1 при p ≥ 0.5.
4. Логаритъмът на шансовете е линеен: ln(p / (1 - p)) = b + wᵀx. Още една
   единица от признака умножава шансовете по eʷ.
5. При независими примери правдоподобието е произведение от вероятностите
   на наблюдаваните класове. Теглата го максимизират, т.е. минимизират
   log loss. Максимумът се търси с градиентно изкачване:
   wⱼ = wⱼ + α · (y - h(x)) · xⱼ, правило със същия вид като LMS. При
   разделими данни максимум няма и регуляризацията пази теглата крайни.
6. Границата z = 0 е линейна, а линиите с еднаква вероятност са успоредни
   на нея.
7. При K класа OvR обучава K двоични класификатора и избира най-големия
   резултат, дори когато положителните области се припокриват или точката
   е извън всички. Softmax превръща K резултата във вероятности със сбор 1,
   а границите между класовете остават прави.

<a id="zadacha"></a>
## Задача

Спам филтърът от [точка 1 "Класове вместо числа"](#t1) е типична задача за
класификация. В упражнение 2, част 4 перцептронът разпознаваше спам SMS
съобщения, но казваше само "спам" или "не". Логистичната регресия дава и
вероятност. С нея прагът за "спам" може да се избере според цената на
грешката: изгубено нормално съобщение е по-лошо от пропуснат спам.

Файлът `data/sms_spam.csv` съдържа колоните `label` (spam или legit) и
`text`.

### 1. Зареждане и Bag-of-Words

Заредете файла и направете етикет 1 за spam и 0 за legit. Разделете
съобщенията на тренировъчни и тестови: 80/20, `random_state=42` и
`stratify`, както в упражнение 2, част 4. `CountVectorizer` от
scikit-learn прави Bag-of-Words като `build_vocabulary` и `vectorize` там:
`fit` научава речника от тренировъчните съобщения, а `transform` брои
думите. С `min_df=2` думите, които се срещат само в едно съобщение, не
влизат в речника. Колко думи има речникът?

In [ ]:
# from sklearn.feature_extraction.text import CountVectorizer
# sms = pd.read_csv("data/sms_spam.csv")
# y_sms = (sms["label"] == "spam").astype(int)
# train_texts, test_texts, y_train_sms, y_test_sms = train_test_split(...)
# vectorizer = CountVectorizer(min_df=2).fit(train_texts)
# X_train_sms, X_test_sms = vectorizer.transform(train_texts), vectorizer.transform(test_texts)
# print("Думи в речника:", ...)

### 2. Логистична регресия и перцептрон

Обучете `LogisticRegression(max_iter=1000)` с регуляризацията по
подразбиране: думите са хиляди и тренировъчните съобщения почти се
разделят без грешка (ДА ОПИТАМЕ 5). Сравнете тестовата точност с
`Perceptron(random_state=42)` и с `DummyClassifier`.

In [ ]:
# spam_model = LogisticRegression(max_iter=1000).fit(X_train_sms, y_train_sms)
# ...
# print("Тестова точност:", ...)

### 3. Думите с най-голямо тегло

Намерете десетте думи с най-голям и десетте с най-малък коефициент.
`vectorizer.get_feature_names_out()` дава думите по реда на колоните. По
колко се умножават шансовете за спам, ако съобщението съдържа веднъж
думата с най-голям коефициент?

In [ ]:
# words = vectorizer.get_feature_names_out()
# coef = spam_model.coef_[0]
# order = np.argsort(coef)
# ...

### 4. Вероятности и праг

Каква е вероятността за спам на тези две съобщения?

- "Congratulations! You have won a FREE prize. Call now to claim"
- "Are we still meeting for lunch tomorrow?"

За тестовите съобщения пребройте колко спам съобщения са хванати и колко
нормални са отбелязани като спам при праг 0.2, 0.5 и 0.9. Кой праг е
най-подходящ за филтър, който не бива да губи нормални съобщения?

In [ ]:
# p_spam = spam_model.predict_proba(X_test_sms)[:, 1]
# for threshold in [0.2, 0.5, 0.9]:
#     ...